# Part 1: Production Matching (Cu, Mo, Li)

Sections 2A-2D: Codelco division mapping, copper production matching, molybdenum production, lithium fixes.

**Depends on:** Part 0

In [9]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 0 ───────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_0.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv = _state["inv"]
links = _state["links"]
comm_col = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS = _state.get("SMELTERS", [])
PORTS = _state.get("PORTS", [])
SMELTER_NAME_MAP = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})
MATCH_DISAMBIGUATION = _state.get("MATCH_DISAMBIGUATION", {})
IRON_MINE_NAMES = _state.get("IRON_MINE_NAMES", [])
ZINC_MINE_NAMES = _state.get("ZINC_MINE_NAMES", [])

print(f"Loaded state from Part 0: {len(inv)} inv rows, {len(links)} link rows")

# ── Shared utility functions ──────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return sorted(matched)  # deterministic ordering across runs


Loaded state from Part 0: 461 inv rows, 1107 link rows


In [10]:

# ── 2A. Codelco division mapping ─────────────────────────────────────────

section_header("2A. CODELCO DIVISION MAPPING")

codelco_divisions = {k: v for k, v in COMPANY_TO_DEPOSIT.items() if k.startswith("División")}
for div_name, terms in codelco_divisions.items():
    extra = CODELCO_EXTRA_SEARCH.get(div_name, [])
    matched_idx = search_inventory(inv, terms + extra)
    if matched_idx:
        print(f"  {div_name}")
        for idx in matched_idx:
            if pd.isna(inv.at[idx, "OPERATOR_NAME"]) or inv.at[idx, "OPERATOR_NAME"] == "":
                inv.at[idx, "OPERATOR_NAME"] = "Codelco"
            print(f"    -> {inv.at[idx, 'FACILITY_NAME']} ({inv.at[idx, 'FACILITY_TYPE']})")
    else:
        print(f"  {div_name} -> NO MATCH")

# ── 2B. Copper production matching ───────────────────────────────────────

section_header("2B. COPPER PRODUCTION MATCHING")

nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production", header=3, index_col=0)
nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in nat.columns]
latest_yr = max(c for c in nat.columns if isinstance(c, int))

commodity_search = {
    "Copper": "COBRE.*Miles de TM", "Molybdenum": "MOLIBDENO.*TM de fino",
    "Gold": "ORO.*Kg de fino", "Silver": "PLATA.*Kg de fino",
    "Iron": "HIERRO.*Miles de TM", "Zinc": "ZINC.*TM de fino",
}
print(f"COCHILCO latest year: {latest_yr}")
for comm, pattern in commodity_search.items():
    match = [r for r in nat.index if re.search(pattern, str(r), re.IGNORECASE)]
    if match:
        print(f"  {comm:<15} {nat.loc[match[0], latest_yr]:>15,.1f}")

cu_co = pd.read_excel(COCHILCO_PATH, sheet_name="B1_Copper_by_Company", header=3, index_col=0)
cu_co.columns = [int(c) if isinstance(c, (int, float)) else c for c in cu_co.columns]

# Idempotent column creation
for col, default in [("COCHILCO_CU_2024_KMT", np.nan), ("COCHILCO_COMPANY", "")]:
    if col not in inv.columns:
        inv[col] = default

cu_matched, cu_matched_prod = 0, 0.0
for company, search_terms in COMPANY_TO_DEPOSIT.items():
    if company not in cu_co.index or latest_yr not in cu_co.columns:
        continue
    prod = cu_co.loc[company, latest_yr]
    if not isinstance(prod, (int, float)) or pd.isna(prod):
        continue
    for term in search_terms:
        mask = inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)
        mine_mask = mask & inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matches = inv[mine_mask] if mine_mask.any() else inv[mask]
        if len(matches) > 0:
            # Disambiguate when multiple mines match the same term
            if company in MATCH_DISAMBIGUATION and len(matches) > 1:
                preferred = MATCH_DISAMBIGUATION[company]
                exact = matches[matches["FACILITY_NAME"] == preferred]
                if len(exact) > 0:
                    matches = exact
            idx = matches.index[0]
            if pd.isna(inv.at[idx, "COCHILCO_CU_2024_KMT"]):
                inv.at[idx, "COCHILCO_CU_2024_KMT"] = prod
                inv.at[idx, "COCHILCO_COMPANY"] = company
                cu_matched += 1
                cu_matched_prod += prod
                print(f"  {company:<35} -> {inv.at[idx, 'FACILITY_NAME']:<35} {prod:>8,.1f} kMT")
            break
    else:
        print(f"  {company:<35} -> NOT MATCHED ({prod:,.1f} kMT)")

# Dynamic total (no hardcoded fallback)
cu_total = cu_co.loc["TOTAL", latest_yr] if "TOTAL" in cu_co.index and latest_yr in cu_co.columns else cu_matched_prod
if cu_total == cu_matched_prod and "TOTAL" not in cu_co.index:
    print(f"  WARNING: 'TOTAL' row not found. Using sum of matched: {cu_total:,.1f} kMT.")
print(f"\nCu matched: {cu_matched} companies, {cu_matched_prod:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_prod/cu_total*100:.1f}%)")

# ── 2C. Molybdenum production ────────────────────────────────────────────

section_header("2C. MOLYBDENUM PRODUCTION MATCHING")

if "COCHILCO_MO_2024_MT" not in inv.columns:
    inv["COCHILCO_MO_2024_MT"] = np.nan

if os.path.exists(COCHILCO_ORIG):
    wb_orig = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)
    ws = wb_orig["Tabla 4.2"]
    rows_raw = list(ws.iter_rows(values_only=True))

    # Find year header row
    yr_row_idx, year_cols = None, {}
    for i, row in enumerate(rows_raw):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2014 < v < 2025) >= 8:
            yr_row_idx = i
            year_cols = {int(v): j for j, v in enumerate(row) if isinstance(v, (int, float)) and 2014 < v < 2025}
            break

    # Parse Mo by company
    mo_by_company, current_company = {}, None
    for i in range(yr_row_idx + 1, len(rows_raw)):
        row = rows_raw[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        if not label or label.startswith("(") or label.startswith("Fuente"):
            continue
        has_data = any(isinstance(row[j], (int, float)) and row[j] != 0
                       for j in year_cols.values() if j < len(row))
        is_sub = "CONCENTRADO" in label or "ÓXIDO" in label

        if has_data and not is_sub:
            vals = {yr: row[col] for yr, col in year_cols.items() if col < len(row) and isinstance(row[col], (int, float))}
            mo_by_company[label] = vals
            current_company = None
        elif not has_data and not is_sub:
            current_company = label
        elif has_data and current_company:
            vals = {yr: row[col] for yr, col in year_cols.items() if col < len(row) and isinstance(row[col], (int, float))}
            if current_company not in mo_by_company:
                mo_by_company[current_company] = {}
            for yr, v in vals.items():
                mo_by_company[current_company][yr] = mo_by_company[current_company].get(yr, 0) + v

    MO_MAP = {
        "Divisiones Chuquicamata y Radomiro Tomic": ["Chuquicamata", "Radomiro Tomic"],
        "División Salvador": ["Salvador"], "División Andina": ["Andina"],
        "División El Teniente": ["El Teniente", "Teniente"],
        "Collahuasi": ["Collahuasi"], "Sierra Gorda": ["Sierra Gorda"],
        "Caserones": ["Caserones"], "Los Pelambres": ["Los Pelambres", "Pelambres"],
        "Anglo American Sur": ["Los Bronces", "Bronces"],
        "Centinela": ["Centinela"], "Spence": ["Spence"],
        "Quebrada Blanca": ["Quebrada Blanca"],
        "Valle Central": ["Valle Central"],  # tailings reprocessor
    }

    mo_matched, mo_matched_prod = 0, 0.0
    for company, search_terms in MO_MAP.items():
        if company not in mo_by_company:
            continue
        prod = mo_by_company[company].get(2024, 0)
        if prod <= 0:
            continue
        matched_idx = search_inventory(inv, search_terms, require_mine=True)
        if matched_idx:
            idx = matched_idx[0]
            if pd.isna(inv.at[idx, "COCHILCO_MO_2024_MT"]):
                inv.at[idx, "COCHILCO_MO_2024_MT"] = prod
                mo_matched += 1
                mo_matched_prod += prod
                print(f"  {company:<50} -> {inv.at[idx, 'FACILITY_NAME']:<30} {prod:>10,.1f} MT")
        else:
            print(f"  {company:<50} -> NOT MATCHED ({prod:,.1f} MT)")

    mo_national = mo_by_company.get("TOTAL", {}).get(2024, mo_matched_prod)
    if mo_national <= 0:
        mo_national = mo_matched_prod
    print(f"\nMo matched: {mo_matched}, {mo_matched_prod:,.1f} / {mo_national:,.1f} MT "
          f"({mo_matched_prod/mo_national*100:.1f}%)")
    wb_orig.close()
else:
    print(f"  Original COCHILCO not found at {COCHILCO_ORIG}, skipping Mo parsing")

# ── 2D. Lithium fixes + link cleanup ────────────────────────────────────

section_header("2D. LITHIUM FIXES AND LINK CLEANUP")

# Fix Potash-listed Salar facilities to include Lithium
potash_li = inv[
    inv["SOURCE"].str.contains("USGS", na=False) &
    inv["PRIMARY_COMMODITY"].str.contains("Potash", case=False, na=False) &
    inv["FACILITY_NAME"].str.contains("Salar|Carmen|Atacama", case=False, na=False)
]
li_fixes = sum(add_commodity(idx, "Lithium", inv, comm_col) for idx in potash_li.index)
for idx in inv[(inv["FACILITY_NAME"] == "Salar de Atacama") &
               inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)].index:
    add_commodity(idx, "Potassium", inv, comm_col)
print(f"  Lithium commodity fixes: {li_fixes} records")

# Rebuild lithium links from scratch (remove old, generate new within 300 km)
STAGE_LI = {"Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
            "Prospect/Project": "extraction", "Mine (USGS)": "extraction"}
inv["_stage"] = inv["FACILITY_TYPE"].map(STAGE_LI)

li_mines = inv[(inv["_stage"] == "extraction") &
               inv[comm_col].str.contains("Lithium", case=False, na=False) &
               inv["LATITUD"].notna()]
li_plants = inv[(inv["_stage"].isna()) &  # non-extraction = processing
                inv[comm_col].str.contains("Lithium", case=False, na=False) &
                inv["LATITUD"].notna()]

# Deduplicate co-located plants before linking (e.g. triple Chemetall Foote entries)
li_plants_dedup = li_plants.copy()
li_plants_dedup["_loc_key"] = (
    li_plants_dedup["LATITUD"].round(2).astype(str) + "_" +
    li_plants_dedup["LONGITUD"].round(2).astype(str)
)
li_plants_dedup = li_plants_dedup.drop_duplicates(subset=["_loc_key"], keep="first")
li_plants_dedup = li_plants_dedup.drop(columns=["_loc_key"])

# Distance caps: 210 km for active mines (Salar de Atacama to Salar del
# Carmen plants is ~200-207 km), 80 km for prospects
new_li_links = []
for _, mrow in li_mines.iterrows():
    is_prospect = "Prospect" in str(mrow["FACILITY_TYPE"])
    max_dist = 80 if is_prospect else 210
    for _, prow in li_plants_dedup.iterrows():
        dist = haversine_km(mrow["LATITUD"], mrow["LONGITUD"], prow["LATITUD"], prow["LONGITUD"])
        if dist <= max_dist:
            new_li_links.append({
                "MINE_NAME": mrow["FACILITY_NAME"], "MINE_TYPE": mrow["FACILITY_TYPE"],
                "MINE_STATUS": mrow.get("STATUS", ""), "MINE_LAT": mrow["LATITUD"],
                "MINE_LON": mrow["LONGITUD"], "MINE_REGION": mrow.get("REGION", ""),
                "MINE_OPERATOR": mrow.get("OPERATOR_NAME", ""),
                "PLANT_NAME": prow["FACILITY_NAME"], "PLANT_TYPE": prow["FACILITY_TYPE"],
                "PLANT_STATUS": prow.get("STATUS", ""), "PLANT_LAT": prow["LATITUD"],
                "PLANT_LON": prow["LONGITUD"], "PLANT_OPERATOR": prow.get("OPERATOR_NAME", ""),
                "PLANT_OWNER": prow.get("OWNER_NAME", ""),
                "PLANT_CAPACITY": prow.get("CAPACITY"),
                "PLANT_CAPACITY_UNITS": prow.get("CAPACITY_UNITS", ""),
                "SHARED_COMMODITIES": "Lithium", "DISTANCE_KM": round(dist, 1),
            })

old_li = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
links = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]

if new_li_links:
    li_df = pd.DataFrame(new_li_links)
    for col in links.columns:
        if col not in li_df.columns:
            li_df[col] = np.nan
    links = pd.concat([links, li_df[links.columns]], ignore_index=True)

# Remove far prospect links and deduplicate by plant location
li_mask = links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)
# Prospect distance already handled above (80 km cap)
li_subset = links[li_mask].copy()
non_li = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]
li_subset["_loc"] = li_subset["PLANT_LAT"].round(2).astype(str) + "_" + li_subset["PLANT_LON"].round(2).astype(str)
li_deduped = li_subset.sort_values("DISTANCE_KM").drop_duplicates(subset=["MINE_NAME", "_loc"], keep="first").drop(columns=["_loc"])
links = pd.concat([non_li, li_deduped], ignore_index=True).sort_values(["MINE_NAME", "DISTANCE_KM"])

inv.drop(columns=["_stage"], inplace=True, errors="ignore")
li_final = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
print(f"  Lithium links: {old_li} old -> {li_final} cleaned")



# ── Save state after Part 1 ─────────────────────────────────────────
_prev_path = os.path.join(DIR_PRELIM, "_pipeline_state_0.pkl")
with open(_prev_path, "rb") as _f:
    _save_state = pickle.load(_f)
_save_state["inv"] = inv
_save_state["links"] = links
_out_path = os.path.join(DIR_PRELIM, "_pipeline_state_1.pkl")
with open(_out_path, "wb") as _f:
    pickle.dump(_save_state, _f)
print(f"State saved to {_out_path}")



2A. CODELCO DIVISION MAPPING
  División El Teniente
    -> El Teniente (Mine (active))
    -> El Teniente plant (Processing Plant)
  División Chuquicamata
    -> Chuquicamata (Mine (active))
    -> Chuquicamata Division (Processing Plant)
    -> Chuquicamata Mine (plant to acid-leach fine copper) (Processing Plant)
    -> Chuquicamata plant (Processing Plant)
    -> Chuquicamata SX-EW plant (oxide) and smelter (Smelter)
  División Radomiro Tomic
    -> Radomiro Tomic (Mine (active))
    -> Radomiro Tomic SX-EW plant (SX-EW Plant)
  División Andina
    -> Andina (Mine (active))
  División Ministro Hales
    -> Ministro Hales (Mine (active))
  División Gabriela Mistral
    -> Gabriela Mistral (Mine (active))
    -> Gabriela Mistral SX-EW plant (SX-EW Plant)
  División Salvador
    -> Salvador (Mine (active))
    -> Potrerillos plant (Processing Plant)
    -> Potrerillos SX-EW refinery and smelter (Smelter)

2B. COPPER PRODUCTION MATCHING
COCHILCO latest year: 2024
  Copper              

# Part 2: Production Matching (Au, Ag, Fe, Zn)

Section 2E: Non-copper mineral production matching from COCHILCO Anuario tables.

**Depends on:** Part 1

In [11]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 1 ───────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_1.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv = _state["inv"]
links = _state["links"]
comm_col = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS = _state.get("SMELTERS", [])
PORTS = _state.get("PORTS", [])
SMELTER_NAME_MAP = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})
MATCH_DISAMBIGUATION = _state.get("MATCH_DISAMBIGUATION", {})
IRON_MINE_NAMES = _state.get("IRON_MINE_NAMES", [])
ZINC_MINE_NAMES = _state.get("ZINC_MINE_NAMES", [])

print(f"Loaded state from Part 1: {len(inv)} inv rows, {len(links)} link rows")

# ── Shared utility functions ──────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return sorted(matched)  # deterministic ordering across runs


Loaded state from Part 1: 461 inv rows, 1109 link rows


In [12]:

# ── 2E. GOLD, SILVER, IRON & ZINC PRODUCTION MATCHING ───────────────────
#
# The COCHILCO Anuario does NOT have company-level tables for Au, Ag, Fe,
# or Zn. Company-level data only exists for Cu (Tabla 2.2) and Mo (4.2).
#
# Strategy: parse regional production from COCHILCO_PATH C_* sheets,
# then allocate to individual mines using Sernageomin reserve/resource
# data (already in the inventory from the scraper's Layer 3 download)
# as the weighting factor.
#
# Weighting priority (per mine):
#   1. Resource tonnage (Gold_Resource, Silver_Resource, etc.)
#   2. Reserve tonnage (Gold_Reserve, Silver_Reserve, etc.)
#
# Mines with neither resource nor reserve data receive weight 0 and are
# excluded from allocation. Only active mines are considered.

section_header("2E. NON-COPPER MINERAL PRODUCTION (Au, Ag, Fe, Zn)")

# ── Column setup (idempotent) ────────────────────────────────────────────

MINERAL_COLS = {
    "Gold": {
        "col": "COCHILCO_AU_2024_KG", "unit": "Kg",
        "pattern": "ORO", "commodity_kw": "Gold",
        "resource_col": "Gold_Resource", "reserve_col": "Gold_Reserve",
    },
    "Silver": {
        "col": "COCHILCO_AG_2024_KG", "unit": "Kg",
        "pattern": "PLATA", "commodity_kw": "Silver",
        "resource_col": "Silver_Resource", "reserve_col": "Silver_Reserve",
    },
    "Iron": {
        "col": "COCHILCO_FE_2024_KMT", "unit": "kMT",
        "pattern": "HIERRO", "commodity_kw": "Iron",
        "resource_col": "Iron_Resource", "reserve_col": "Iron_Reserve",
    },
    "Zinc": {
        "col": "COCHILCO_ZN_2024_MT", "unit": "MT",
        "pattern": "ZINC", "commodity_kw": "Zinc",
        "resource_col": "Zinc_Resource", "reserve_col": "Zinc_Reserve",
    },
}

for cfg in MINERAL_COLS.values():
    if cfg["col"] not in inv.columns:
        inv[cfg["col"]] = np.nan

# ── Parse regional production from C_* sheets ───────────────────────────

REGION_SHEET_MAP = {
    "C_AricaParinacota": ["Arica", "Parinacota"],
    "C_Tarapaca": ["Tarapacá", "Tarapaca"],
    "C_Antofagasta": ["Antofagasta"],
    "C_Atacama": ["Atacama"],
    "C_Coquimbo": ["Coquimbo"],
    "C_Valparaiso": ["Valparaíso", "Valparaiso"],
    "C_Santiago": ["Metropolitana", "Santiago", "Región Metropolitana"],
    "C_OHiggins": ["O'Higgins", "OHiggins", "Libertador"],
    "C_Maule": ["Maule"],
    "C_BioBio": ["Biobío", "BioBio", "Bio-Bio"],
    "C_Araucania": ["Araucanía", "Araucania"],
    "C_LosLagos": ["Los Lagos"],
    "C_Aysen": ["Aysén", "Aysen", "Ibáñez"],
    "C_Magallanes": ["Magallanes"],
}

wb_path = openpyxl.load_workbook(COCHILCO_PATH, read_only=True, data_only=True)

regional_production = {}  # {mineral: {sheet_name: value}}

for sheet_name in wb_path.sheetnames:
    if not sheet_name.startswith("C_") or sheet_name == "C0_Cu_Regional_Product":
        continue
    ws = wb_path[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    yr_col = None
    for j, v in enumerate(rows[3] if len(rows) > 3 else []):
        if isinstance(v, (int, float)) and int(v) == 2024:
            yr_col = j
            break
    if yr_col is None:
        continue

    for i in range(4, min(15, len(rows))):
        row = rows[i]
        label = str(row[0]).upper() if row[0] else ""
        val = row[yr_col] if yr_col < len(row) else None
        if not isinstance(val, (int, float)) or val <= 0:
            continue

        for mineral, cfg in MINERAL_COLS.items():
            if cfg["pattern"] in label:
                regional_production.setdefault(mineral, {})[sheet_name] = val

wb_path.close()

# ── Helper: build region mask ────────────────────────────────────────────

def _region_mask(region_patterns):
    mask = pd.Series(False, index=inv.index)
    for pat in region_patterns:
        mask |= inv["REGION"].str.contains(pat, case=False, na=False)
    return mask


# ── Helper: compute weights for a set of candidate mines ─────────────────

def _compute_weights(candidates, cfg):
    """
    Build a weight array for candidate mines using only resource/reserve data.
    Mines with neither return weight 0 and will be excluded from allocation.
    Priority: resource tonnage > reserve tonnage.
    """
    res_col = cfg["resource_col"]
    rev_col = cfg["reserve_col"]
    weights = np.zeros(len(candidates))

    for i, idx in enumerate(candidates.index):
        res_val = inv.at[idx, res_col] if res_col in inv.columns else np.nan
        if pd.notna(res_val) and res_val > 0:
            weights[i] = res_val
            continue

        rev_val = inv.at[idx, rev_col] if rev_col in inv.columns else np.nan
        if pd.notna(rev_val) and rev_val > 0:
            weights[i] = rev_val

    return weights


# ── Main allocation loop ─────────────────────────────────────────────────

for mineral, cfg in MINERAL_COLS.items():
    col_name = cfg["col"]
    unit = cfg["unit"]
    commodity_kw = cfg["commodity_kw"]
    res_col = cfg["resource_col"]
    rev_col = cfg["reserve_col"]
    regions = regional_production.get(mineral, {})

    if not regions:
        print(f" {mineral}: no regional production data found")
        continue

    total_regional = sum(regions.values())
    print(f" {mineral}: {len(regions)} producing regions, total {total_regional:,.1f} {unit}")

    total_assigned = 0.0
    for sheet_name, regional_val in regions.items():
        region_patterns = REGION_SHEET_MAP.get(sheet_name, [])
        if not region_patterns:
            continue

        rmask = _region_mask(region_patterns)
        commodity_mask = inv[comm_col].str.contains(commodity_kw, case=False, na=False)
        active_mask = inv["FACILITY_TYPE"].str.contains("Mine.*active", case=True, na=False, regex=True)
        candidates = inv[rmask & commodity_mask & active_mask & pd.isna(inv[col_name])]

        region_label = sheet_name.replace("C_", "")

        if len(candidates) == 0:
            print(f"    {region_label:<25} {regional_val:>10,.1f} {unit}  -> NO active mines")
            continue

        weights = _compute_weights(candidates, cfg)

        # Exclude mines with no resource or reserve data
        valid_mask = weights > 0
        if not valid_mask.any():
            skipped = ", ".join(inv.at[idx, "FACILITY_NAME"] for idx in candidates.index)
            print(f"    {region_label:<25} {regional_val:>10,.1f} {unit}  -> SKIPPED (no resource/reserve data): {skipped}")
            continue

        candidates = candidates[valid_mask]
        weights = weights[valid_mask]
        weight_sum = weights.sum()

        detail_parts = []
        for i_c, idx in enumerate(candidates.index):
            share = regional_val * (weights[i_c] / weight_sum)
            inv.at[idx, col_name] = share
            add_commodity(idx, commodity_kw, inv, comm_col)
            total_assigned += share

            name = inv.at[idx, "FACILITY_NAME"]
            res_val = inv.at[idx, res_col] if res_col in inv.columns else np.nan
            src = "resource" if (pd.notna(res_val) and res_val > 0) else "reserve"
            pct = weights[i_c] / weight_sum * 100
            detail_parts.append(f"{name}({src},{pct:.0f}%)")

        print(f"    {region_label:<25} {regional_val:>10,.1f} {unit}  -> {len(candidates)} mines: "
              f"{', '.join(detail_parts)}")

    pct = (total_assigned / total_regional * 100) if total_regional > 0 else 0
    print(f"    TOTAL assigned: {total_assigned:,.1f} / {total_regional:,.1f} {unit} ({pct:.1f}%)")


# ── Summary ──────────────────────────────────────────────────────────────

section_header("  NON-COPPER PRODUCTION SUMMARY")

nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production", header=3, index_col=0)
nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in nat.columns]

for mineral, cfg in MINERAL_COLS.items():
    col = cfg["col"]
    res_col = cfg["resource_col"]
    rev_col = cfg["reserve_col"]
    n = inv[col].notna().sum()
    total_inv = inv[col].sum() if n > 0 else 0

    nat_total = None
    for idx_label in nat.index:
        if cfg["pattern"] in str(idx_label).upper():
            nat_total = nat.loc[idx_label, 2024] if 2024 in nat.columns else None
            break

    assigned = inv[inv[col].notna()]
    n_res = sum(
        1 for idx in assigned.index
        if res_col in inv.columns and pd.notna(inv.at[idx, res_col]) and inv.at[idx, res_col] > 0
    )
    n_rev = n - n_res

    nat_str = f"{nat_total:,.1f}" if nat_total else "N/A"
    print(f"  {mineral:<10} {n:>3} facilities, {total_inv:>12,.1f} {cfg['unit']}  "
          f"(national: {nat_str} {cfg['unit']})  "
          f"[{n_res} resource-weighted, {n_rev} reserve-weighted]")

if "COCHILCO_MO_2024_MT" in inv.columns:
    n_mo = inv["COCHILCO_MO_2024_MT"].notna().sum()
    total_mo = inv["COCHILCO_MO_2024_MT"].sum()
    print(f"  {'Molybd.':<10} {n_mo:>3} facilities, {total_mo:>12,.1f} MT  [Anuario (company)]")



# ── Save state after Part 2 ─────────────────────────────────────────
_prev_path = os.path.join(DIR_PRELIM, "_pipeline_state_1.pkl")
with open(_prev_path, "rb") as _f:
    _save_state = pickle.load(_f)
_save_state["inv"] = inv
_save_state["links"] = links
_out_path = os.path.join(DIR_PRELIM, "_pipeline_state_2.pkl")
with open(_out_path, "wb") as _f:
    pickle.dump(_save_state, _f)
print(f"State saved to {_out_path}")


2E. NON-COPPER MINERAL PRODUCTION (Au, Ag, Fe, Zn)
 Gold: 8 producing regions, total 29,610.7 Kg
    Antofagasta                 17,948.3 Kg  -> 6 mines: Amancaya(resource,0%), Centinela(resource,67%), El Abra(resource,12%), El Peñón(resource,6%), Guanaco(resource,2%), Sierra Gorda(resource,14%)
    Atacama                      2,962.0 Kg  -> 4 mines: Candelaria(resource,42%), La Coipa(resource,10%), Mantoverde(resource,23%), Salares Norte(resource,25%)
    Coquimbo                     2,961.7 Kg  -> 5 mines: Andacollo Oro(resource,16%), Carmen de Andacollo(resource,10%), El Espino(resource,4%), Los Pelambres(resource,70%), Mantos de Punitaqui(resource,0%)
    Valparaiso                   2,492.0 Kg  -> 1 mines: Pullalli(resource,100%)
    Santiago                     2,492.0 Kg  -> 1 mines: Alhué(resource,100%)
    OHiggins                       720.7 Kg  -> NO active mines
    Aysen                           31.0 Kg  -> 2 mines: Cerro Bayo(resource,37%), El Toqui(resource,63%)
    M